
# Cart-pole direct transcription in PyDrake

This notebook mirrors the MATLAB/SNOPT direct-transcription formulation, but uses PyDrake's `MathematicalProgram` directly.

The important changes are:

- the common sample time `h` is a decision variable with bounds,
- the dynamics constraint is no longer forward Euler,
- each defect uses a Drake fixed-step integrator:

\[
    x_{k+1} - \phi_h(x_k,u_k) = 0,
\]

where \(\phi_h\) is obtained by integrating the parsed `undamped_cartpole.urdf` system over one interval while holding \(u_k\) constant.


In [ ]:

import numpy as np
from matplotlib import pyplot as plt

from pydrake.all import (
    AddMultibodyPlantSceneGraph,
    DiagramBuilder,
    MathematicalProgram,
    Parser,
    RungeKutta2Integrator,
    Solve,
)

from underactuated import ConfigureParser



## Build the cart-pole from Drake's URDF

This is intentionally the same construction pattern as the cart-pole balancing notebook: create a `DiagramBuilder`, add a continuous-time `MultibodyPlant` and `SceneGraph`, parse `package://underactuated/models/undamped_cartpole.urdf`, and finalize the plant.


In [ ]:

# Start construction site of our block diagram.
builder = DiagramBuilder()

# Instantiate the cart-pole and the scene graph.
cartpole, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
parser = Parser(cartpole)
ConfigureParser(parser)
parser.AddModelsFromUrl("package://underactuated/models/undamped_cartpole.urdf")
cartpole.Finalize()

diagram = builder.Build()

# State convention used by the underactuated cart-pole notebooks:
# x = [cart position, pole angle, cart velocity, pole angular velocity].
x0 = np.array([0.0, 0.0, 0.0, 0.0])
xd = np.array([0.0, np.pi, 0.0, 0.0])

context = diagram.CreateDefaultContext()
plant_context = cartpole.GetMyMutableContextFromRoot(context)
plant_context.get_mutable_continuous_state_vector().SetFromVector(xd)
cartpole.get_actuation_input_port().FixValue(plant_context, [0.0])



## One-step flow map using a Drake integrator

For direct transcription, we want the same arithmetic path for every NLP evaluation. So we use `IntegrateWithSingleFixedStepToTime(h)` instead of an adaptive call.

I picked `RungeKutta2Integrator`: it is still explicit and cheap enough for many NLP constraint evaluations, but it is a meaningful upgrade over the first-order Euler defect. For this notebook, each interval is one fixed RK2 step from time `0` to time `h`.


In [ ]:

def integrate_cartpole_one_step(xk, uk, hk):
    """Return phi_h(xk, uk) using Drake's fixed-step RK2 integrator.

    The input is held constant over the interval [0, h]. This function is
    intentionally numerical because it will be used inside a MathematicalProgram
    nonlinear constraint callback.
    """
    hk = float(hk)
    uk = np.asarray(uk, dtype=float).reshape(1)
    xk = np.asarray(xk, dtype=float).reshape(4)

    local_context = diagram.CreateDefaultContext()
    local_plant_context = cartpole.GetMyMutableContextFromRoot(local_context)

    local_context.SetTime(0.0)
    local_plant_context.get_mutable_continuous_state_vector().SetFromVector(xk)
    cartpole.get_actuation_input_port().FixValue(local_plant_context, uk)

    integrator = RungeKutta2Integrator(diagram, hk, local_context)
    integrator.set_fixed_step_mode(True)
    integrator.Initialize()
    integrator.IntegrateWithSingleFixedStepToTime(hk)

    final_context = integrator.get_context()
    final_plant_context = cartpole.GetMyContextFromRoot(final_context)
    return final_plant_context.get_continuous_state_vector().CopyToVector()


# Smoke test: at the upright equilibrium with zero input, the state should remain nearly fixed.
print(integrate_cartpole_one_step(xd, [0.0], 0.05))



## Direct transcription with `h` as a decision variable

The MATLAB/SNOPT version used a decision vector like

\[
    w = [h, X(:), U(:)],
\]

but the bounds made `h` effectively fixed. Here `h` is genuinely free inside `[h_min, h_max]`.

The dynamics defect is

\[
    x_{k+1} - \phi_h(x_k,u_k) = 0.
\]


In [ ]:

# Number of knot points. Increase this after the notebook is working.
N = 81
nX = 4
nU = 1

# This is the key change: h is a true NLP decision variable with bounds.
h_min = 0.02
h_max = 0.08
h_guess = 0.05

# Bounds similar in spirit to the MATLAB version.
x_min = np.array([-10.0, -10.0, -10.0, -10.0])
x_max = np.array([ 10.0,  10.0,  10.0,  10.0])
u_min = np.array([-100.0])
u_max = np.array([ 100.0])

# A slightly loose terminal box is much friendlier for the first solve.
# Tighten this after the formulation is converging.
terminal_tol = np.array([2e-2, 2e-2, 5e-2, 5e-2])
xf_min = xd - terminal_tol
xf_max = xd + terminal_tol

Q = np.diag([0.01, 0.01, 0.01, 0.01])
Qf = np.diag([1.0, 50.0, 1.0, 1.0])
R = np.diag([0.01])

time_weight = 1.0


In [ ]:

def quadratic_form(z, Q):
    return sum(Q[i, j] * z[i] * z[j] for i in range(Q.shape[0]) for j in range(Q.shape[1]))


def make_dynamics_defect_constraint():
    def defect(vars):
        xk = vars[0:4]
        uk = vars[4:5]
        xkp1 = vars[5:9]
        hk = vars[9]
        return xkp1 - integrate_cartpole_one_step(xk, uk, hk)

    return defect


In [ ]:

prog = MathematicalProgram()

# Decision variables: X is 4 x N, U is 1 x (N-1), h is scalar.
x = prog.NewContinuousVariables(nX, N, "x")
u = prog.NewContinuousVariables(nU, N - 1, "u")
h = prog.NewContinuousVariables(1, "h")[0]

# Initial / terminal state constraints.
prog.AddBoundingBoxConstraint(x0, x0, x[:, 0])
prog.AddBoundingBoxConstraint(xf_min, xf_max, x[:, -1])

# Bounds on h, state, and input.
prog.AddBoundingBoxConstraint(h_min, h_max, [h])
for k in range(N):
    prog.AddBoundingBoxConstraint(x_min, x_max, x[:, k])
for k in range(N - 1):
    prog.AddBoundingBoxConstraint(u_min, u_max, u[:, k])

# Dynamics constraints: x_{k+1} - phi_h(x_k, u_k) = 0.
defect = make_dynamics_defect_constraint()
for k in range(N - 1):
    vars_k = np.concatenate((x[:, k], u[:, k], x[:, k + 1], np.array([h])))
    prog.AddConstraint(defect, np.zeros(nX), np.zeros(nX), vars_k)

# Running cost: h * sum_k (x_k - xd)'Q(x_k - xd) + u_k'R u_k.
for k in range(N - 1):
    e = x[:, k] - xd
    prog.AddCost(h * (quadratic_form(e, Q) + quadratic_form(u[:, k], R)))

# Terminal cost.
eN = x[:, -1] - xd
prog.AddCost(quadratic_form(eN, Qf))

# Like the minimum-time direct-collocation example, add final-time pressure.
# Set time_weight = 0 above if you want the pure MATLAB-style quadratic objective only.
prog.AddCost(time_weight * (N - 1) * h)

# Initial guess: linear state interpolation, zero controls, middle of h bounds.
for i in range(nX):
    prog.SetInitialGuess(x[i, :], np.linspace(x0[i], xd[i], N))
prog.SetInitialGuess(u, np.zeros((nU, N - 1)))
prog.SetInitialGuess(h, h_guess)



## Solve

This follows the `double_integrator.ipynb` style: formulate the problem directly as a `MathematicalProgram`, then call `Solve(prog)`. Since the dynamics defect is a Python nonlinear callback through a Drake integrator, Drake will treat this as a nonlinear program.


In [ ]:

result = Solve(prog)
print(result.get_solver_id().name())
print(result.is_success())
print(result.get_solution_result())
print("optimal h =", result.GetSolution(h))
print("total time =", (N - 1) * result.GetSolution(h))


In [ ]:

x_sol = result.GetSolution(x)
u_sol = result.GetSolution(u)
h_sol = result.GetSolution(h)
tx = np.arange(N) * h_sol
tu = np.arange(N - 1) * h_sol

print("final state =", x_sol[:, -1])


In [ ]:

plt.figure()
plt.plot(x_sol[0, :], x_sol[1, :], "-o", markersize=2)
plt.plot(x0[0], x0[1], "go", label="start")
plt.plot(xd[0], xd[1], "ro", label="target")
plt.xlabel("cart position")
plt.ylabel("pole angle")
plt.title("Cart-pole swing-up trajectory")
plt.grid(True)
plt.legend()

plt.figure()
plt.plot(tx, x_sol[0, :])
plt.xlabel("time [s]")
plt.ylabel("cart position")
plt.grid(True)

plt.figure()
plt.plot(tx, x_sol[1, :])
plt.xlabel("time [s]")
plt.ylabel("pole angle")
plt.grid(True)

plt.figure()
plt.step(tu, u_sol[0, :], where="post")
plt.xlabel("time [s]")
plt.ylabel("cart force")
plt.grid(True)

plt.figure()
plt.plot(x_sol[1, :], x_sol[3, :])
plt.xlabel("pole angle")
plt.ylabel("angular velocity")
plt.title("Angle phase portrait")
plt.grid(True)
plt.show()



## Optional sanity check: compare one optimized interval

This evaluates the first optimized defect numerically. It should be close to zero when the solve succeeds.


In [ ]:

k = 0
x_next_integrated = integrate_cartpole_one_step(x_sol[:, k], u_sol[:, k], h_sol)
print("defect at k=0 =", x_sol[:, k + 1] - x_next_integrated)
print("norm =", np.linalg.norm(x_sol[:, k + 1] - x_next_integrated))
